# Yambda AutoFE — запуск в Google Colab (GPU)

**Runtime → Change runtime type → T4 GPU**, затем Run all. Результаты пишутся на Google Drive (`yambda-autofe-results`).

Протокол: два AutoFE-поиска — (а) агрегаты (≈2750 кандидатов) поверх ручной сессии, (б) AutoFE-full: агрегаты + грамматика сессии (≈2830 кандидатов), фиксирован только контекст. Поиск выполняется на 1000 пользователях; найденный набор признаков без повторного поиска переносится на 5000 пользователей (проверка переносимости и экономия памяти/времени). Обучение моделей — XGBoost на GPU.

In [ ]:
N_USERS = 5000   # большой прогон; AutoFE-поиск делается на 1000 пользователях и переносится сюда

import os, glob, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/yambda-autofe-results'
os.makedirs(OUT, exist_ok=True)

# Код: git clone (если репозиторий публичный) -> иначе yambda-autofe.zip с Google Drive -> иначе загрузить zip вручную
REPO = 'https://github.com/georgiken/yambda-autofe.git'
os.chdir('/content')  # при повторном запуске cwd может быть удаляемой папкой /content/recsys
subprocess.run('rm -rf /content/recsys', shell=True)
ok = subprocess.run(f'GIT_TERMINAL_PROMPT=0 git clone -q {REPO} /content/recsys', shell=True).returncode == 0
if not ok:
    zips = glob.glob('/content/drive/MyDrive/**/yambda-autofe*.zip', recursive=True)
    if zips:
        z = zips[0]
    else:
        print('Репозиторий приватный: выберите файл yambda-autofe.zip из папки recsys')
        from google.colab import files
        z = '/content/' + next(iter(files.upload()))
    zipfile.ZipFile(z).extractall('/content/recsys')
    print('код взят из', z)
%cd /content/recsys
!pip install -q -r requirements.txt 2>&1 | tail -2
!nproc; free -g

In [ ]:
# 0) проверка GPU: XGBoost должен обучаться на cuda
!nvidia-smi --query-gpu=name,memory.total --format=csv
import numpy as np, xgboost as xgb
X = np.random.rand(20000, 20); y = (X[:, 0] + np.random.rand(20000) > 1).astype(int)
b = xgb.train({'tree_method': 'hist', 'device': 'cuda', 'objective': 'binary:logistic'}, xgb.QuantileDMatrix(X, y), 20)
print('xgboost', xgb.__version__, 'GPU OK:', b.save_config().count('cuda') > 0)

In [ ]:
# 1) тесты на утечки (~15 с)
!python -m pytest -q tests

In [ ]:
# EDA (~2-4 мин): статистика и графики -> Drive/yambda-autofe-results/eda
!python scripts/eda.py --n-users {N_USERS or 10000} --out {OUT}/eda 2>&1 | grep -v Warning | head -80
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'{OUT}/eda/*.png')): display(Image(f))

In [ ]:
# 1) 1000 пользователей: ручной бейзлайн + два AutoFE-поиска + все контроли (~12-15 мин)
!python scripts/run_experiments.py --n-users 1000 --backend xgb --device cuda --out {OUT}/u1000 2>&1 | grep -v Warning

In [ ]:
# 2) 5000 пользователей: те же модели, AutoFE-наборы перенесены из u1000 (~15 мин)
main = 'u10000' if N_USERS is None else f'u{N_USERS}'
flag = '' if N_USERS is None else f'--n-users {N_USERS}'
!python scripts/run_experiments.py {flag} --autofe-from {OUT}/u1000 --backend xgb --device cuda --out {OUT}/{main} 2>&1 | grep -v Warning

In [ ]:
# 3) значимость: paired bootstrap по пользователям
!python scripts/bootstrap.py {OUT}/u1000 --n 1000 | tail -45
!python scripts/bootstrap.py {OUT}/{main} --n 1000 | tail -45

In [ ]:
# 4) таблицы и графики + архив
!rm -f {OUT}/events_*.parquet
!python scripts/make_report.py {OUT} > /dev/null
!cd {OUT} && zip -qr /content/results.zip . && ls -la /content/results.zip
from google.colab import files; files.download('/content/results.zip')